# Skin Lesion Classification — Transfer Learning Comparison
### ISIC Skin Cancer (9 classes) — for Task 01

This notebook reproduces the three comparison tables from the assignment, using the
**Skin Cancer ISIC (9 classes)** dataset instead of HAM10000:

- **Table 1** — Transfer-learning backbones (AlexNet, VGG16, VGG19, ResNet18, ResNet50,
  ResNet101, DenseNet121, EfficientNet-B0), each fine-tuned end-to-end as a plain
  classifier (no dilation — the paper's dilation trick is architecture-specific and not
  required by the task sheet's table, which only asks for standard transfer-learning
  metrics).
- **Table 2** — Deep features from one strong backbone (ResNet50) fed into classical
  classifiers (Logistic Regression, Decision Tree, Random Forest, KNN, Linear SVM,
  RBF-SVM, XGBoost).
- **Table 3** — Computational efficiency of each backbone (parameter count, model size
  in MB, FLOPs, inference time, and its Table-1 accuracy).


## 0. Install dependencies

In [ ]:
!pip install -q kagglehub thop xgboost scikit-learn --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 63.4 MB/s eta 0:00:00


## 1. Download the dataset

The dataset is **Skin Cancer ISIC (9 classes)**:
https://www.kaggle.com/datasets/nodoubttome/skin-cancer9-classesisic



In [ ]:
import kagglehub

# Downloads the dataset and returns the local path to the extracted folder
dataset_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Dataset downloaded to:", dataset_path)

import os
for root, dirs, files in os.walk(dataset_path):
    level = root.replace(dataset_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    if level >= 2:
        dirs[:] = []  # don't recurse further, just show top structure
]


Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Dataset downloaded to: /kaggle/input/skin-cancer9-classesisic
skin-cancer9-classesisic/
  Skin cancer ISIC The International Skin Imaging Collaboration/
    Test/
    Train/


## 2. Build a file list and stratified train/val/test split

The Kaggle download usually contains `Train/` and `Test/` folders, each with one
subfolder per class. We pool everything together and re-split it ourselves
80% / 10% / 10% (stratified by class), matching the paper's methodology, so the
validation set is not accidentally the same as the original Test folder.


In [ ]:
import glob, os
import pandas as pd
from sklearn.model_selection import train_test_split

IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

all_files = []
for ext in IMG_EXTS:
    all_files.extend(glob.glob(os.path.join(dataset_path, "**", f"*{ext}"), recursive=True))
    all_files.extend(glob.glob(os.path.join(dataset_path, "**", f"*{ext.upper()}"), recursive=True))

print("Total images found:", len(all_files))

# Class name = parent folder name (works for .../Train/<class>/img.jpg or .../Test/<class>/img.jpg)
records = [{"path": f, "label": os.path.basename(os.path.dirname(f))} for f in all_files]
df = pd.DataFrame(records)
print(df["label"].value_counts())

classes = sorted(df["label"].unique())
num_classes = len(classes)
class_to_idx = {c: i for i, c in enumerate(classes)}
print("\nClasses:", classes)
print("Num classes:", num_classes)

df["label_idx"] = df["label"].map(class_to_idx)

# Stratified 80/10/10 split
train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df["label_idx"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label_idx"], random_state=42)

print(f"\nTrain: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")


Total images found: 2357
label
pigmented benign keratosis    478
melanoma                      454
basal cell carcinoma          392
nevus                         373
squamous cell carcinoma       197
vascular lesion               142
actinic keratosis             130
dermatofibroma                111
seborrheic keratosis           80
Name: count, dtype: int64

Classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Num classes: 9

Train: 1885  Val: 236  Test: 236


## 3. Dataset class, transforms, and dataloaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

IMG_SIZE = 224  # standard input size for all torchvision backbones used below
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomAffine(degrees=0, translate=(0.2, 0.2), shear=10, scale=(0.8, 1.2)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SkinLesionDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, row["label_idx"]

train_ds = SkinLesionDataset(train_df, train_tf)
val_ds = SkinLesionDataset(val_df, eval_tf)
test_ds = SkinLesionDataset(test_df, eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("Batches -> train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))


Using device: cuda
Batches -> train: 59 val: 8 test: 8


## 4. Model builder  (all eight Table-1 backbones)


In [ ]:
import torch.nn as nn
from torchvision import models

def get_model(name, num_classes):
    """Return an ImageNet-pretrained backbone with its final layer replaced
    for `num_classes`-way classification."""
    name = name.lower()

    if name == "alexnet":
        m = models.alexnet(weights=models.AlexNet_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)

    elif name == "vgg16":
        m = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)

    elif name == "vgg19":
        m = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        m.classifier[6] = nn.Linear(m.classifier[6].in_features, num_classes)

    elif name == "resnet18":
        m = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)

    elif name == "resnet50":
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)

    elif name == "resnet101":
        m = models.resnet101(weights=models.ResNet101_Weights.IMAGENET1K_V1)
        m.fc = nn.Linear(m.fc.in_features, num_classes)

    elif name == "densenet121":
        m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)

    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unknown model name: {name}")

    return m

MODEL_NAMES = ["alexnet", "vgg16", "vgg19", "resnet18", "resnet50",
               "resnet101", "densenet121", "efficientnet_b0"]


## 5. Training and evaluation routine

In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score)
import time

def train_model(model, train_loader, val_loader, epochs=10, lr=1e-4, freeze_epochs=2):
    """Fine-tune: freeze backbone for `freeze_epochs`, then unfreeze everything
    and continue training (mirrors the paper's feature-extraction -> fine-tune approach)."""
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()

    # Phase 1: freeze all but the final classifier layer
    for p in model.parameters():
        p.requires_grad = False
    # unfreeze just the last layer(s) — works for all architectures above since we
    # replaced classifier/fc last
    last_params = list(model.parameters())[-2:]
    for p in last_params:
        p.requires_grad = True

    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    for epoch in range(freeze_epochs):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()

    # Phase 2: unfreeze everything, fine-tune with a smaller LR
    for p in model.parameters():
        p.requires_grad = True
    optimizer = torch.optim.Adam(model.parameters(), lr=lr * 0.1)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min",
                                                            factor=0.1, patience=3)

    best_val_loss = float("inf")
    best_state = None
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            out = model(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        train_loss = running_loss / len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                loss = criterion(out, labels)
                val_loss += loss.item() * imgs.size(0)
        val_loss /= len(val_loader.dataset)
        scheduler.step(val_loss)

        print(f"  epoch {epoch+1}/{epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model


def evaluate_model(model, data_loader, num_classes):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, labels in data_loader:
            imgs = imgs.to(DEVICE)
            out = model(imgs)
            probs = torch.softmax(out, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
            all_probs.extend(probs)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)

    acc = accuracy_score(all_labels, all_preds) * 100
    prec = precision_score(all_labels, all_preds, average="weighted", zero_division=0) * 100
    rec = recall_score(all_labels, all_preds, average="weighted", zero_division=0) * 100
    f1 = f1_score(all_labels, all_preds, average="weighted", zero_division=0) * 100
    try:
        auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="weighted") * 100
    except ValueError:
        auc = float("nan")  # can happen if a class is missing from a small test split

    return {"Accuracy": acc, "Precision": prec, "Recall": rec, "F1-Score": f1, "AUC": auc}


## 6. Train & evaluate all eight backbones → **Table 1**

This runs each model in `MODEL_NAMES` through fine-tuning and reports the metrics



In [ ]:
EPOCHS = 10  # increase (e.g. 20-30) for a more polished final result

table1_results = {}

for name in MODEL_NAMES:
    print(f"\n=== Training {name} ===")
    torch.cuda.empty_cache()
    model = get_model(name, num_classes)
    model = train_model(model, train_loader, val_loader, epochs=EPOCHS)
    metrics = evaluate_model(model, test_loader, num_classes)
    table1_results[name] = metrics
    print(f"  {name} test metrics: {metrics}")

    # keep ResNet50's fine-tuned weights around for the Table-2 feature extractor
    if name == "resnet50":
        resnet50_finetuned = model

    del model
    torch.cuda.empty_cache()

table1_df = pd.DataFrame(table1_results).T
table1_df = table1_df.round(2)
table1_df.index.name = "Model"
print("\n=== TABLE 1 ===")
table1_df



=== Training alexnet ===
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 183MB/s]


  epoch 1/10  train_loss=1.5049  val_loss=1.4713
  epoch 2/10  train_loss=1.3597  val_loss=1.3709
  epoch 3/10  train_loss=1.2438  val_loss=1.2706
  epoch 4/10  train_loss=1.1494  val_loss=1.2327
  epoch 5/10  train_loss=1.1101  val_loss=1.1658
  epoch 6/10  train_loss=1.0573  val_loss=1.1507
  epoch 7/10  train_loss=1.0030  val_loss=1.1610
  epoch 8/10  train_loss=0.9824  val_loss=1.1223
  epoch 9/10  train_loss=0.9416  val_loss=1.1098
  epoch 10/10  train_loss=0.9346  val_loss=1.1035
  alexnet test metrics: {'Accuracy': 66.52542372881356, 'Precision': 66.05535332591869, 'Recall': 66.52542372881356, 'F1-Score': 64.8607977167699, 'AUC': 92.68174969259138}

=== Training vgg16 ===
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:08<00:00, 65.8MB/s]


  epoch 1/10  train_loss=1.6429  val_loss=1.5332
  epoch 2/10  train_loss=1.3448  val_loss=1.3004
  epoch 3/10  train_loss=1.1946  val_loss=1.1967
  epoch 4/10  train_loss=1.1112  val_loss=1.1819
  epoch 5/10  train_loss=1.0217  val_loss=1.2385
  epoch 6/10  train_loss=0.9618  val_loss=1.1299
  epoch 7/10  train_loss=0.9073  val_loss=1.1192
  epoch 8/10  train_loss=0.8698  val_loss=1.1707
  epoch 9/10  train_loss=0.8530  val_loss=1.0722
  epoch 10/10  train_loss=0.7775  val_loss=1.0876
  vgg16 test metrics: {'Accuracy': 66.94915254237289, 'Precision': 66.30022067738767, 'Recall': 66.94915254237289, 'F1-Score': 65.26289705557838, 'AUC': 92.5199753240928}

=== Training vgg19 ===
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...

100%|██████████| 30.8M/30.8M [00:00<00:00, 176MB/s]


  epoch 1/10  train_loss=1.8382  val_loss=1.8059
  epoch 2/10  train_loss=1.6793  val_loss=1.6772
  epoch 3/10  train_loss=1.5444  val_loss=1.5431
  epoch 4/10  train_loss=1.4087  val_loss=1.4210
  epoch 5/10  train_loss=1.2879  val_loss=1.3246
  epoch 6/10  train_loss=1.1927  val_loss=1.2425
  epoch 7/10  train_loss=1.1266  val_loss=1.1763
  epoch 8/10  train_loss=1.0599  val_loss=1.1138
  epoch 9/10  train_loss=1.0079  val_loss=1.0789
  epoch 10/10  train_loss=0.9448  val_loss=1.0298
  densenet121 test metrics: {'Accuracy': 71.1864406779661, 'Precision': 70.53093108400871, 'Recall': 71.1864406779661, 'F1-Score': 69.85307438856742, 'AUC': 93.91924398627937}

=== Training efficientnet_b0 ===
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 158MB/s]


  epoch 1/10  train_loss=1.8677  val_loss=1.8625
  epoch 2/10  train_loss=1.8079  val_loss=1.8098
  epoch 3/10  train_loss=1.7491  val_loss=1.7583
  epoch 4/10  train_loss=1.6964  val_loss=1.6898
  epoch 5/10  train_loss=1.6371  val_loss=1.6386
  epoch 6/10  train_loss=1.5741  val_loss=1.5758
  epoch 7/10  train_loss=1.5110  val_loss=1.5273
  epoch 8/10  train_loss=1.4517  val_loss=1.4697
  epoch 9/10  train_loss=1.4083  val_loss=1.4212
  epoch 10/10  train_loss=1.3473  val_loss=1.3742
  efficientnet_b0 test metrics: {'Accuracy': 58.89830508474576, 'Precision': 57.86651023631614, 'Recall': 58.89830508474576, 'F1-Score': 53.92606292843082, 'AUC': 89.3426538932331}

=== TABLE 1 ===


,Accuracy,Precision,Recall,F1-Score,AUC
Model,,,,,
alexnet,66.53,66.06,66.53,64.86,92.68
vgg16,66.95,66.30,66.95,65.26,92.52
vgg19,67.37,67.62,67.37,66.45,93.22
resnet18,72.88,70.61,72.88,71.05,94.35
resnet50,74.58,73.34,74.58,73.40,95.23
resnet101,76.69,74.74,76.69,75.45,95.79
densenet121,71.19,70.53,71.19,69.85,93.92
efficientnet_b0,58.90,57.87,58.90,53.93,89.34


In [ ]:
table1_df.to_csv("table1_transfer_learning_models.csv")
table1_df


,Accuracy,Precision,Recall,F1-Score,AUC
Model,,,,,
alexnet,66.53,66.06,66.53,64.86,92.68
vgg16,66.95,66.30,66.95,65.26,92.52
vgg19,67.37,67.62,67.37,66.45,93.22
resnet18,72.88,70.61,72.88,71.05,94.35
resnet50,74.58,73.34,74.58,73.40,95.23
resnet101,76.69,74.74,76.69,75.45,95.79
densenet121,71.19,70.53,71.19,69.85,93.92
efficientnet_b0,58.90,57.87,58.90,53.93,89.34


## 7. Deep feature extraction + classical classifiers → **Table 2**

We use the fine-tuned **ResNet50** from Table 1 as a fixed feature extractor
(remove the final FC layer, take the 2048-d pooled features), then train seven
classical classifiers on top of those features.


In [ ]:
from torch import nn as _nn

def build_feature_extractor(finetuned_resnet50):
    fe = _nn.Sequential(*list(finetuned_resnet50.children())[:-1])  # drop final fc
    fe.eval().to(DEVICE)
    return fe

feature_extractor = build_feature_extractor(resnet50_finetuned)

def extract_features(loader, extractor):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(DEVICE)
            out = extractor(imgs).squeeze(-1).squeeze(-1)  # (B, 2048)
            feats.append(out.cpu().numpy())
            labels.append(lbls.numpy())
    return np.concatenate(feats), np.concatenate(labels)

X_train, y_train = extract_features(train_loader, feature_extractor)
X_val, y_val = extract_features(val_loader, feature_extractor)
X_test, y_test = extract_features(test_loader, feature_extractor)

# fold validation features into training for the classical classifiers
X_train_full = np.concatenate([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

print("Feature shapes:", X_train_full.shape, X_test.shape)


Feature shapes: (2121, 2048) (236, 2048)


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_full)
X_test_scaled = scaler.transform(X_test)

classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=2000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(n_neighbors=5),
    "Linear SVM": SVC(kernel="linear", probability=True, random_state=42),
    "RBF-SVM": SVC(kernel="rbf", probability=True, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=300, use_label_encoder=False,
                              eval_metric="mlogloss", random_state=42, n_jobs=-1),
}

table2_results = {}
for clf_name, clf in classifiers.items():
    print(f"Training {clf_name} ...")
    clf.fit(X_train_scaled, y_train_full)
    preds = clf.predict(X_test_scaled)
    probs = clf.predict_proba(X_test_scaled)

    acc = accuracy_score(y_test, preds) * 100
    prec = precision_score(y_test, preds, average="weighted", zero_division=0) * 100
    rec = recall_score(y_test, preds, average="weighted", zero_division=0) * 100
    f1 = f1_score(y_test, preds, average="weighted", zero_division=0) * 100
    try:
        auc = roc_auc_score(y_test, probs, multi_class="ovr", average="weighted") * 100
    except ValueError:
        auc = float("nan")

    table2_results[clf_name] = {"Feature Extractor": "Deep Features (ResNet50)",
                                 "Accuracy": acc, "Precision": prec, "Recall": rec,
                                 "F1-Score": f1, "AUC": auc}

table2_df = pd.DataFrame(table2_results).T
table2_df = table2_df[["Feature Extractor", "Accuracy", "Precision", "Recall", "F1-Score", "AUC"]]
for col in ["Accuracy", "Precision", "Recall", "F1-Score", "AUC"]:
    table2_df[col] = table2_df[col].astype(float).round(2)
table2_df.index.name = "Classifier"
print("\n=== TABLE 2 ===")
table2_df

Training Logistic Regression ...
Training Decision Tree ...
Training Random Forest ...
Training K-Nearest Neighbors (KNN) ...
Training Linear SVM ...


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Training RBF-SVM ...


/usr/local/lib/python3.13/dist-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Training XGBoost ...


/usr/local/lib/python3.13/dist-packages/xgboost/training.py:200: UserWarning: [05:31:47] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)



=== TABLE 2 ===


,Feature Extractor,Accuracy,Precision,Recall,F1-Score,AUC
Classifier,,,,,,
Logistic Regression,Deep Features (ResNet50),69.07,69.14,69.07,68.21,93.47
Decision Tree,Deep Features (ResNet50),57.20,57.90,57.20,57.31,75.03
Random Forest,Deep Features (ResNet50),75.42,74.10,75.42,74.37,94.59
K-Nearest Neighbors (KNN),Deep Features (ResNet50),74.15,76.08,74.15,74.33,91.15
Linear SVM,Deep Features (ResNet50),72.03,71.61,72.03,71.41,94.61
RBF-SVM,Deep Features (ResNet50),76.27,74.68,76.27,75.09,94.65
XGBoost,Deep Features (ResNet50),75.85,74.46,75.85,74.86,94.00


In [ ]:
table2_df.to_csv("table2_classifier_comparison.csv")
table2_df


,Feature Extractor,Accuracy,Precision,Recall,F1-Score,AUC
Classifier,,,,,,
Logistic Regression,Deep Features (ResNet50),69.07,69.14,69.07,68.21,93.47
Decision Tree,Deep Features (ResNet50),57.20,57.90,57.20,57.31,75.03
Random Forest,Deep Features (ResNet50),75.42,74.10,75.42,74.37,94.59
K-Nearest Neighbors (KNN),Deep Features (ResNet50),74.15,76.08,74.15,74.33,91.15
Linear SVM,Deep Features (ResNet50),72.03,71.61,72.03,71.41,94.61
RBF-SVM,Deep Features (ResNet50),76.27,74.68,76.27,75.09,94.65
XGBoost,Deep Features (ResNet50),75.85,74.46,75.85,74.86,94.00


## 8. Computational efficiency → **Table 3**

For each backbone we report: parameter count (millions), model size on disk (MB),
FLOPs (GigaFLOPs) for one 224x224 forward pass, average inference time per image
(measured on the current device), and the Table-1 test accuracy.


In [ ]:
from thop import profile
import tempfile

def compute_efficiency(model_name, num_classes, n_warmup=5, n_runs=30):
    model = get_model(model_name, num_classes).to(DEVICE)
    model.eval()

    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)

    # Params
    n_params = sum(p.numel() for p in model.parameters())

    # FLOPs
    macs, _ = profile(model, inputs=(dummy,), verbose=False)
    gflops = (2 * macs) / 1e9  # MACs -> FLOPs (x2), then to Giga

    # Model size on disk
    with tempfile.NamedTemporaryFile(suffix=".pt") as tmp:
        torch.save(model.state_dict(), tmp.name)
        size_mb = os.path.getsize(tmp.name) / (1024 ** 2)

    # Inference time (batch size 1, average over n_runs after warmup)
    with torch.no_grad():
        for _ in range(n_warmup):
            _ = model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.time()
        for _ in range(n_runs):
            _ = model(dummy)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed_ms = (time.time() - start) / n_runs * 1000

    del model
    torch.cuda.empty_cache()

    return {
        "Parameters (M)": n_params / 1e6,
        "Model Size (MB)": size_mb,
        "FLOPs (G)": gflops,
        "Inference Time (ms)": elapsed_ms,
    }

table3_results = {}
for name in MODEL_NAMES:
    print(f"Profiling {name} ...")
    eff = compute_efficiency(name, num_classes)
    eff["Accuracy (%)"] = table1_df.loc[name, "Accuracy"]
    table3_results[name] = eff

table3_df = pd.DataFrame(table3_results).T
table3_df = table3_df[["Parameters (M)", "Model Size (MB)", "FLOPs (G)",
                        "Inference Time (ms)", "Accuracy (%)"]].round(2)
table3_df.index.name = "Model"
print("\n=== TABLE 3 ===")
table3_df


Profiling alexnet ...
Profiling vgg16 ...
Profiling vgg19 ...
Profiling resnet18 ...
Profiling resnet50 ...
Profiling resnet101 ...
Profiling densenet121 ...
Profiling efficientnet_b0 ...

=== TABLE 3 ===


,Parameters (M),Model Size (MB),FLOPs (G),Inference Time (ms),Accuracy (%)
Model,,,,,
alexnet,57.04,217.60,1.42,2.11,66.53
vgg16,134.30,512.32,30.93,11.32,66.95
vgg19,139.61,532.57,39.26,13.58,67.37
resnet18,11.18,42.73,3.65,3.80,72.88
resnet50,23.53,90.06,8.26,5.99,74.58
resnet101,42.52,162.82,15.73,17.46,76.69
densenet121,6.96,27.18,5.79,21.88,71.19
efficientnet_b0,4.02,15.71,0.83,7.89,58.90


In [ ]:
table3_df.to_csv("table3_computational_efficiency.csv")
table3_df


NameError: name 'table3_df' is not defined

## 9. Summary (all three tables)




In [ ]:
print("TABLE 1 — Transfer Learning Models\n", table1_df, "\n")
print("TABLE 2 — Classifier Comparison\n", table2_df, "\n")
print("TABLE 3 — Computational Efficiency\n", table3_df, "\n")

from google.colab import files
files.download("table1_transfer_learning_models.csv")
files.download("table2_classifier_comparison.csv")
files.download("table3_computational_efficiency.csv")


TABLE 1 — Transfer Learning Models
                  Accuracy  Precision  Recall  F1-Score    AUC
Model                                                        
alexnet             66.53      66.06   66.53     64.86  92.68
vgg16               66.95      66.30   66.95     65.26  92.52
vgg19               67.37      67.62   67.37     66.45  93.22
resnet18            72.88      70.61   72.88     71.05  94.35
resnet50            74.58      73.34   74.58     73.40  95.23
resnet101           76.69      74.74   76.69     75.45  95.79
densenet121         71.19      70.53   71.19     69.85  93.92
efficientnet_b0     58.90      57.87   58.90     53.93  89.34 

TABLE 2 — Classifier Comparison
                                   Feature Extractor  Accuracy  Precision  \
Classifier                                                                 
Logistic Regression        Deep Features (ResNet50)     69.07      69.14   
Decision Tree              Deep Features (ResNet50)     57.20      57.90   
Rando

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>